In [ ]:
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm import tqdm

# Load sampled complaints
df = pd.read_csv("../data/processed/sampled_complaints.csv")

print(f"Dataset Shape: {df.shape}")

# Chunking Configuration
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

all_chunks = []
all_metadata = []

print("Creating chunks...")

for idx, row in tqdm(df.iterrows(), total=len(df)):

    text = str(row["clean_narrative"])

    chunks = splitter.split_text(text)

    for chunk in chunks:

        all_chunks.append(chunk)

        all_metadata.append(
            {
                "complaint_id": str(idx),
                "product": row["Product"]
            }
        )

print(f"Total Chunks Created: {len(all_chunks)}")

# Load Embedding Model
print("Loading embedding model...")

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# Generate Embeddings
print("Generating embeddings...")

embeddings = model.encode(
    all_chunks,
    show_progress_bar=True
)

print("Embedding Shape:", embeddings.shape)

# Create ChromaDB Vector Store
print("Creating ChromaDB vector store...")

client = chromadb.PersistentClient(
    path="../vector_store/chroma_db"
)

collection = client.get_or_create_collection(
    name="complaints"
)

ids = [str(i) for i in range(len(all_chunks))]

collection.add(
    ids=ids,
    documents=all_chunks,
    embeddings=embeddings.tolist(),
    metadatas=all_metadata
)

print("Vector Store Successfully Created!")

# Test Retrieval
query = "Credit card payment issue"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("\nSample Retrieval Results:\n")

for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}")
    print("-" * 50)
    print(doc[:500])